# 🥙 LSTM on Recipe Data

In this notebook, we'll walk through the steps required to train your own LSTM on the recipes dataset

In [1]:
import numpy as np
import json
import re
import string

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, losses

## 0. Parameters <a name="parameters"></a>

In [2]:
VOCAB_SIZE = 10000
MAX_LEN = 200
EMBEDDING_DIM = 100
N_UNITS = 128
VALIDATION_SPLIT = 0.2
SEED = 42
LOAD_MODEL = False
BATCH_SIZE = 32
EPOCHS = 25

## 1. Load the data <a name="load"></a>

In [5]:
from google.colab import files
import json

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

with open(file_name) as json_data:
    recipe_data = json.load(json_data)

print("JSON file loaded successfully")

Saving full_format_recipes.json to full_format_recipes.json
JSON file loaded successfully


In [6]:
# Filter the dataset
filtered_data = [
    "Recipe for " + x["title"] + " | " + " ".join(x["directions"])
    for x in recipe_data
    if "title" in x
    and x["title"] is not None
    and "directions" in x
    and x["directions"] is not None
]

In [7]:
# Count the recipes
n_recipes = len(filtered_data)
print(f"{n_recipes} recipes loaded")

20111 recipes loaded


In [8]:
example = filtered_data[9]
print(example)

Recipe for Ham Persillade with Mustard Potato Salad and Mashed Peas  | Chop enough parsley leaves to measure 1 tablespoon; reserve. Chop remaining leaves and stems and simmer with broth and garlic in a small saucepan, covered, 5 minutes. Meanwhile, sprinkle gelatin over water in a medium bowl and let soften 1 minute. Strain broth through a fine-mesh sieve into bowl with gelatin and stir to dissolve. Season with salt and pepper. Set bowl in an ice bath and cool to room temperature, stirring. Toss ham with reserved parsley and divide among jars. Pour gelatin on top and chill until set, at least 1 hour. Whisk together mayonnaise, mustard, vinegar, 1/4 teaspoon salt, and 1/4 teaspoon pepper in a large bowl. Stir in celery, cornichons, and potatoes. Pulse peas with marjoram, oil, 1/2 teaspoon pepper, and 1/4 teaspoon salt in a food processor to a coarse mash. Layer peas, then potato salad, over ham.


## 2. Tokenise the data

In [9]:
# Pad the punctuation, to treat them as separate 'words'
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s


text_data = [pad_punctuation(x) for x in filtered_data]

In [10]:
# Display an example of a recipe
example_data = text_data[9]
example_data

'Recipe for Ham Persillade with Mustard Potato Salad and Mashed Peas | Chop enough parsley leaves to measure 1 tablespoon ; reserve . Chop remaining leaves and stems and simmer with broth and garlic in a small saucepan , covered , 5 minutes . Meanwhile , sprinkle gelatin over water in a medium bowl and let soften 1 minute . Strain broth through a fine - mesh sieve into bowl with gelatin and stir to dissolve . Season with salt and pepper . Set bowl in an ice bath and cool to room temperature , stirring . Toss ham with reserved parsley and divide among jars . Pour gelatin on top and chill until set , at least 1 hour . Whisk together mayonnaise , mustard , vinegar , 1 / 4 teaspoon salt , and 1 / 4 teaspoon pepper in a large bowl . Stir in celery , cornichons , and potatoes . Pulse peas with marjoram , oil , 1 / 2 teaspoon pepper , and 1 / 4 teaspoon salt in a food processor to a coarse mash . Layer peas , then potato salad , over ham . '

In [11]:
# Convert to a Tensorflow Dataset
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000)
)

In [12]:
# Create a vectorisation layer
vectorize_layer = layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1,
)

In [13]:
# Adapt the layer to the training set
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()

In [14]:
# Display some token:word mappings
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")

0: 
1: [UNK]
2: .
3: ,
4: and
5: to
6: in
7: the
8: with
9: a


In [15]:
# Display the same example converted to ints
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy())

[  26   16  557    1    8  298  335  189    4 1054  494   27  332  228
  235  262    5  594   11  133   22  311    2  332   45  262    4  671
    4   70    8  171    4   81    6    9   65   80    3  121    3   59
   12    2  299    3   88  650   20   39    6    9   29   21    4   67
  529   11  164    2  320  171  102    9  374   13  643  306   25   21
    8  650    4   42    5  931    2   63    8   24    4   33    2  114
   21    6  178  181 1245    4   60    5  140  112    3   48    2  117
  557    8  285  235    4  200  292  980    2  107  650   28   72    4
  108   10  114    3   57  204   11  172    2   73  110  482    3  298
    3  190    3   11   23   32  142   24    3    4   11   23   32  142
   33    6    9   30   21    2   42    6  353    3 3224    3    4  150
    2  437  494    8 1281    3   37    3   11   23   15  142   33    3
    4   11   23   32  142   24    6    9  291  188    5    9  412  572
    2  230  494    3   46  335  189    3   20  557    2    0    0    0
    0 

## 3. Create the Training Set

In [16]:
# Create the training set of recipes and the same text shifted by one word
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)
    tokenized_sentences = vectorize_layer(text)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y


train_ds = text_ds.map(prepare_inputs)

## 4. Build the LSTM <a name="build"></a>

In [17]:
inputs = layers.Input(shape=(None,), dtype="int32")
x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x = layers.LSTM(N_UNITS, return_sequences=True)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
lstm = models.Model(inputs, outputs)
lstm.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, None, 100)      │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 128)      │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 10000)    │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,407,248 (9.18 MB)

 Trainable params: 2,407,248 (9.18 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
if LOAD_MODEL:
    # model.load_weights('./models/model')
    lstm = models.load_model("./models/lstm", compile=False)

## 5. Train the LSTM <a name="train"></a>

In [19]:
loss_fn = losses.SparseCategoricalCrossentropy()
lstm.compile("adam", loss_fn)

In [20]:
# Create a TextGenerator checkpoint
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }  # <1>

    def sample_from(self, probs, temperature):  # <2>
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]  # <3>
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:  # <4>
            x = np.array([start_tokens])
            y = self.model.predict(x, verbose=0)  # <5>
            sample_token, probs = self.sample_from(y[0][-1], temperature)  # <6>
            info.append({"prompt": start_prompt, "word_probs": probs})
            start_tokens.append(sample_token)  # <7>
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("recipe for", max_tokens=100, temperature=1.0)

In [21]:
# Create a model save checkpoint
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint/checkpoint.weights.h5",
    save_weights_only=True,
    save_freq="epoch",
    verbose=0,
)

tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

# Tokenize starting prompt
text_generator = TextGenerator(vocab)

In [22]:
lstm.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[model_checkpoint_callback, tensorboard_callback, text_generator],
)

Epoch 1/25
629/629 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 5.0079
generated text:
recipe for chocolate lemon with upper onion for chicken casserole | a breads water into you remaining pan overhang to yolk to a pat foil in shallow caldillo . cover . cookies inches in a large bowl ; beat water in small bowl and let 2 minutes or breast and wide tablespoon 1 over paste with / frittata to celery to mixture ; cool stew . transfer scraping out beets . add an barley glaze and in small 12 cup and reserve 1 to large small bowl ) . rice magic , until softened in slotted ancho sides 

629/629 ━━━━━━━━━━━━━━━━━━━━ 19s 26ms/step - loss: 4.1553
Epoch 2/25
628/629 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 3.1033
generated text:
recipe for sturdy grilled chipotle cilantro | in a bowl together 1 tablespoon butter first oil with medium - high heat , about 5 minutes . let rice in a baking 9 - glacé skillet over , stirring occasionally , about 1 teaspoon 1 hour . remove pan and biscuits onto room tempera

In [26]:
# Save the final model
import os
os.makedirs("./models", exist_ok=True)
lstm.save("./models/lstm.keras")

## 6. Generate text using the LSTM

In [27]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        print(f"\nPROMPT: {i['prompt']}")
        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, i in zip(p_sorted, i_sorted):
            print(f"{vocab[i]}:   \t{np.round(100*p,2)}%")
        print("--------\n")

In [28]:
info = text_generator.generate(
    "recipe for roasted vegetables | chop 1 /", max_tokens=10, temperature=1.0
)


generated text:
recipe for roasted vegetables | chop 1 / 2 cup



In [29]:
print_probs(info, vocab)


PROMPT: recipe for roasted vegetables | chop 1 /
4:   	45.06999969482422%
2:   	31.489999771118164%
8:   	11.0%
3:   	10.59000015258789%
1:   	0.5299999713897705%
--------


PROMPT: recipe for roasted vegetables | chop 1 / 2
cup:   	48.45000076293945%
teaspoon:   	12.699999809265137%
inch:   	7.409999847412109%
-:   	6.110000133514404%
pound:   	2.7100000381469727%
--------



In [30]:
info = text_generator.generate(
    "recipe for roasted vegetables | chop 1 /", max_tokens=10, temperature=0.2
)


generated text:
recipe for roasted vegetables | chop 1 / 4 cup



In [31]:
print_probs(info, vocab)


PROMPT: recipe for roasted vegetables | chop 1 /
4:   	85.61000061035156%
2:   	14.25%
8:   	0.07000000029802322%
3:   	0.05999999865889549%
1:   	0.0%
--------


PROMPT: recipe for roasted vegetables | chop 1 / 4
cup:   	99.97000122070312%
inch:   	0.019999999552965164%
teaspoon:   	0.009999999776482582%
-:   	0.009999999776482582%
pound:   	0.0%
--------



In [34]:
info = text_generator.generate(
    "recipe for chocolate ice cream |", max_tokens=25, temperature=0.5
)
print_probs(info, vocab)


generated text:
recipe for chocolate ice cream | stir together sugar , butter , sugar , and salt in a medium bowl . let stand until cool


PROMPT: recipe for chocolate ice cream |
combine:   	28.68000030517578%
bring:   	17.969999313354492%
preheat:   	15.850000381469727%
in:   	11.520000457763672%
whisk:   	10.9399995803833%
--------


PROMPT: recipe for chocolate ice cream | stir
together:   	64.7699966430664%
sugar:   	12.90999984741211%
cream:   	9.109999656677246%
chocolate:   	7.300000190734863%
first:   	3.25%
--------


PROMPT: recipe for chocolate ice cream | stir together
sugar:   	34.650001525878906%
all:   	21.700000762939453%
butter:   	9.779999732971191%
cream:   	8.529999732971191%
flour:   	6.690000057220459%
--------


PROMPT: recipe for chocolate ice cream | stir together sugar
,:   	60.470001220703125%
and:   	39.52000045776367%
in:   	0.0%
with:   	0.0%
(:   	0.0%
--------


PROMPT: recipe for chocolate ice cream | stir together sugar ,
butter:   	50.61000061035156

In [35]:
info = text_generator.generate(
    "recipe for chocolate ice cream |", max_tokens=25, temperature=1.4
)
print_probs(info, vocab)


generated text:
recipe for chocolate ice cream | combine blueberry wine and half of apricots , blackberries , strawberries , horseradish extract food coloring cornstarch and blend


PROMPT: recipe for chocolate ice cream |
combine:   	8.270000457763672%
bring:   	7.0%
preheat:   	6.690000057220459%
in:   	5.96999979019165%
whisk:   	5.860000133514404%
--------


PROMPT: recipe for chocolate ice cream | combine
first:   	6.269999980926514%
all:   	4.630000114440918%
cream:   	4.059999942779541%
1:   	2.7699999809265137%
flour:   	2.319999933242798%
--------


PROMPT: recipe for chocolate ice cream | combine blueberry
sauce:   	5.389999866485596%
juice:   	4.909999847412109%
nectar:   	4.260000228881836%
ingredients:   	3.6500000953674316%
syrup:   	3.549999952316284%
--------


PROMPT: recipe for chocolate ice cream | combine blueberry wine
,:   	51.02000045776367%
and:   	21.790000915527344%
ingredients:   	3.200000047683716%
in:   	1.5700000524520874%
or:   	1.2799999713897705%
-----